In [56]:
import numpy as np
from enum import IntEnum
from typing import Optional
from glob import glob
import matplotlib.pyplot as plt
import open3d as o3d

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
class CameraDir(IntEnum):
    POS_X = 0
    NEG_X = 1
    POS_Y = 2
    NEG_Y = 3
    POS_Z = 4


_CAMERA_CONFIGS = {
    CameraDir.POS_X: (
        np.array([1, 0, 0], dtype=np.float32),
        np.array([0, 1, 0], dtype=np.float32),
        np.array([0, 0, 1], dtype=np.float32),
    ),
    CameraDir.NEG_X: (
        np.array([-1, 0, 0], dtype=np.float32),
        np.array([0, 1, 0], dtype=np.float32),
        np.array([0, 0, -1], dtype=np.float32),
    ),
    CameraDir.POS_Y: (
        np.array([0, 1, 0], dtype=np.float32),
        np.array([1, 0, 0], dtype=np.float32),
        np.array([0, 0, 1], dtype=np.float32),
    ),
    CameraDir.NEG_Y: (
        np.array([0, -1, 0], dtype=np.float32),
        np.array([-1, 0, 0], dtype=np.float32),
        np.array([0, 0, 1], dtype=np.float32),
    ),
    CameraDir.POS_Z: (
        np.array([0, 0, 1], dtype=np.float32),
        np.array([1, 0, 0], dtype=np.float32),
        np.array([0, 1, 0], dtype=np.float32),
    ),
}

In [48]:
def depth_maps_to_pointcloud(
    depth_maps: list[np.ndarray],
    camera_origins: Optional[list[np.ndarray]] = None,
    depth_scale: float = 1.0,
    invalid_depth: float = 0.0,
) -> np.ndarray:
    assert len(depth_maps) == 5

    directions = list(CameraDir)
    all_points = []

    for i, (depth_map, cam_dir) in enumerate(zip(depth_maps, directions)):
        depth = np.squeeze(depth_map.astype(np.float32) * depth_scale)
        H, W = depth.shape

        forward, right, up = _CAMERA_CONFIGS[cam_dir]

        u = np.linspace(-1.0, 1.0, W, dtype=np.float32)
        v = np.linspace(-1.0, 1.0, H, dtype=np.float32)
        uu, vv = np.meshgrid(u, v)

        mask = depth != invalid_depth
        d = depth[mask]
        uu_m = uu[mask]
        vv_m = vv[mask]

        origin = camera_origins[i] if camera_origins is not None else np.zeros(3, dtype=np.float32)

        points = (
            origin[np.newaxis, :]
            + d[:, np.newaxis] * forward[np.newaxis, :]
            + uu_m[:, np.newaxis] * right[np.newaxis, :]
            + vv_m[:, np.newaxis] * up[np.newaxis, :]
        )

        all_points.append(points)

    return np.concatenate(all_points, axis=0)

In [49]:
camera_origins = [
    np.array([ 0,  0,  5.0], dtype=np.float32),
    np.array([ 0,  4,  0.1], dtype=np.float32),
    np.array([ 0, -4,  0.1], dtype=np.float32),
    np.array([ 4,  0,  0.1], dtype=np.float32),
    np.array([-4,  0,  0.1], dtype=np.float32),
]

In [50]:
directory = sorted(glob("../samples_of_each_environment/high_clutter/depth*"))
images = []

for img in directory:
    images.append(plt.imread(img)[:, :, [0]])


In [53]:
points = depth_maps_to_pointcloud(images, camera_origins)

In [58]:
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)  # points shape: (N, 3)
o3d.visualization.draw_geometries([pcd])

[Open3D WARNING] GLFW Error: Failed to detect any supported platform
[Open3D WARNING] GLFW initialized for headless rendering.
[Open3D WARNING] Failed to initialize GLEW.
[Open3D WARNING] [DrawGeometries] Failed creating OpenGL window.
